# 02 — Geocode container addresses

This notebook takes the cleaned LIMASAM addresses and geocodes each one using Nominatim (OpenStreetMap), producing point coordinates for spatial analysis.

**Input:** `data/interim/containers_cleaned.csv` — 478 cleaned addresses tagged by structure.

**Output:** `data/processed/containers_geocoded.csv` — same rows with `lat`, `lon`, `geocoding_quality`, and `geocoding_method` columns.

## Strategy

The first attempts revealed three challenges that shaped the fallback chain:

1. **Street-name variants on OpenStreetMap.** OSM often records streets without `de la` / `del` (`Avenida Paloma` instead of `Avenida de la Paloma`). When the original address fails, the geocoder retries with a simplified version.
2. **Mid-address district names confuse the geocoder.** Strings like `..., 26, Distrito Centro, 29009, ...` fail because Nominatim treats the district token as part of the street. The geocoder removes it on retry.
3. **Intersections.** ~13% of LIMASAM addresses describe street intersections rather than house numbers. Nominatim handles these inconsistently, so the geocoder splits the intersection and tries each street alone as a fallback.

Each result is tagged with a **quality** score:

- **`high`** — the original (or near-original) query returned a `house`/`building` match, or a `street_only` query returned its street
- **`medium`** — only a street-level match was returned for a `house_number` query (the house number was lost)
- **`low`** — only one side of an intersection was matched, or the house number had to be stripped
- **`failed`** — no result after all fallbacks

All results are restricted to a generous Málaga bounding box to avoid false matches in other Spanish cities with similar street names.

A persistent cache (`data/interim/geocoding_cache.json`) saves every result, so re-running this notebook is instant — only new addresses are geocoded.

In [1]:
import re
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_INTERIM / "containers_cleaned.csv"
OUTPUT_PATH = DATA_PROCESSED / "containers_geocoded.csv"
CACHE_PATH = DATA_INTERIM / "geocoding_cache.json"

print(f"Input:  {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"Cache:  {CACHE_PATH}")

Input:  c:\Users\user\Documents\Projects\malaga-textile-access\data\interim\containers_cleaned.csv
Output: c:\Users\user\Documents\Projects\malaga-textile-access\data\processed\containers_geocoded.csv
Cache:  c:\Users\user\Documents\Projects\malaga-textile-access\data\interim\geocoding_cache.json


In [2]:
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} addresses")
print(f"Columns: {df.columns.tolist()}")
print(f"\nAddress type distribution:")
print(df["address_type"].value_counts())
df.head(3)

Loaded 478 addresses
Columns: ['address_raw', 'address_type', 'address_clean']

Address type distribution:
address_type
house_number    381
intersection     64
street_only      33
Name: count, dtype: int64


,address_raw,address_type,address_clean
0,"Av. de la Paloma, 36-38, Carretera de Cádiz, 2...",house_number,"Avenida de la Paloma, 36-38, Carretera de Cádi..."
1,"Av. de la Paloma, 43, Carretera de Cádiz, 2900...",house_number,"Avenida de la Paloma, 43, Carretera de Cádiz, ..."
2,"C. Conde de Guadalhorce, 6-8, Cruz de Humillad...",house_number,"Calle Conde de Guadalhorce, 6-8, Cruz de Humil..."


## Geocoder setup

Configure Nominatim with:
- A custom user-agent (Nominatim's usage policy requires identification)
- A rate limiter (max 1 request per ~1 second)
- A Málaga bounding box (strict — addresses matched outside this box are rejected)

In [3]:
# Initialize Nominatim with required user-agent
geolocator = Nominatim(
    user_agent="malaga-textile-access-project (lidavaynberg@gmail.com)",
    timeout=10,
)

# Rate-limited wrapper (Nominatim's policy: max ~1 req/sec)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.1,
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=False,
)

# Málaga bounding box. Note: viewbox expects (latitude, longitude) tuples.
# A generous box covering the municipal area + immediate surroundings.
MALAGA_BBOX = [(36.65, -4.55), (36.78, -4.30)]
#                ↑ SW (lat, lon)   ↑ NE (lat, lon)


def geocode_in_malaga(address: str):
    """Geocode an address, restricting results to Málaga's bounding box and Spain."""
    return geocode(
        address,
        country_codes="es",
        viewbox=MALAGA_BBOX,
        bounded=True,
    )

print("Geocoder configured. Rate limit: 1 request / 1.1s.")
print(f"Bounding box: SW {MALAGA_BBOX[0]} → NE {MALAGA_BBOX[1]}")

Geocoder configured. Rate limit: 1 request / 1.1s.
Bounding box: SW (36.65, -4.55) → NE (36.78, -4.3)


In [4]:
# Quick sanity check on three addresses of different types
test_cases = [
    "Calle Goya, 7, Málaga, Spain",
    "Andarax and Camino de San Rafael, Málaga, Spain",
    "Calle Almogía, Málaga, Spain",
]

print("Sanity check — geocoding 3 sample addresses:\n")
for addr in test_cases:
    loc = geocode_in_malaga(addr)
    if loc:
        print(f"  ✓ {addr}")
        print(f"    → ({loc.latitude:.5f}, {loc.longitude:.5f})  type={loc.raw.get('type')}")
    else:
        print(f"  ✗ {addr}  NOT FOUND (but fallbacks may still succeed)")
    print()

Sanity check — geocoding 3 sample addresses:

  ✓ Calle Goya, 7, Málaga, Spain
    → (36.70553, -4.43741)  type=house

  ✗ Andarax and Camino de San Rafael, Málaga, Spain  NOT FOUND (but fallbacks may still succeed)

  ✓ Calle Almogía, Málaga, Spain
    → (36.71424, -4.45314)  type=secondary



## Fallback chain

The `geocode_with_fallbacks` function tries up to 8 strategies in order, stopping at the first match. Helper functions below transform the input address in different ways before each retry.

In [5]:
def simplify_address(address: str) -> str:
    """Remove 'de la', 'del', 'de' between street type and street name."""
    return re.sub(
        r"\b(Avenida|Calle|Plaza|Camino|Carretera|Pasaje|Paseo|Glorieta|Carril|Alameda|Cuesta|Travesía|Pje)\s+(de\s+la|de\s+las|del|de\s+los|de)\s+",
        r"\1 ",
        address,
        flags=re.IGNORECASE,
    )


def expand_name_abbreviations(s: str) -> str:
    """Expand abbreviations inside street names (not just street types)."""
    rules = [
        (r"\bSgto\.?",          "Sargento"),
        (r"\bGral\.?",          "General"),
        (r"\bDr\.?",            "Doctor"),
        (r"\bDra\.?",           "Doctora"),
        (r"\bNtra\.?\s*Sra\.?", "Nuestra Señora"),
        (r"\bSta\.?",           "Santa"),
        (r"\bS\.\s+(?=[A-ZÁÉÍÓÚÑ])", "San "),
    ]
    for pattern, repl in rules:
        s = re.sub(pattern, repl, s, flags=re.IGNORECASE)
    return s


def strip_district(address: str) -> str:
    """Remove a district name between house number and postal code.
    Example: 'Calle X, 26, Distrito Centro, 29009, Málaga, Spain'
           → 'Calle X, 26, 29009, Málaga, Spain'
    """
    return re.sub(
        r"(,\s*\d+[\d\-A-Za-z]*),\s*[^,\d]+(?=,\s*\d{5}\b)",
        r"\1",
        address,
    )


def strip_house_number(address: str) -> str:
    """Remove the first numeric token (house number) from an address."""
    return re.sub(r",?\s*\d+[\d\-A-Za-z]*\s*,", ",", address, count=1).strip(", ")


def reduce_house_range(address: str) -> str:
    """Convert house number ranges like '13-11' or '6-8' to just the first number."""
    return re.sub(r"(,\s*)(\d+)-\d+", r"\g<1>\g<2>", address)


def split_intersection(address: str):
    """Split 'Street A and Street B, ..., Málaga, Spain' into two single-street queries."""
    parts = re.split(r"\s+and\s+", address, maxsplit=1, flags=re.IGNORECASE)
    if len(parts) != 2:
        return None, None
    street_a = parts[0].strip()
    street_b_raw = parts[1].strip()
    # Strip leading number from street B (e.g. '11 and Calle X' → 'Calle X')
    street_b_raw = re.sub(r"^\d+[\d\-A-Za-z]*\s*", "", street_b_raw)
    street_b_name = street_b_raw.split(",")[0].strip()
    return f"{street_a}, Málaga, Spain", f"{street_b_name}, Málaga, Spain"


# Manual replacements for typos discovered during testing
MANUAL_TYPO_FIXES = {
    "Petersen": "Peterson",
    "Herrea":   "Herrera",
}

def apply_typo_fixes(address: str) -> str:
    """Apply known manual typo corrections to an address."""
    for wrong, right in MANUAL_TYPO_FIXES.items():
        address = re.sub(rf"\b{wrong}\b", right, address, flags=re.IGNORECASE)
    return address

In [6]:
def _classify_quality(loc, address_type: str, fallback_used: bool) -> str:
    """Return quality tag based on what Nominatim returned and how we got there."""
    is_house_match = (loc.raw.get("class") == "place" and loc.raw.get("type") == "house")
    
    if is_house_match:
        return "high"
    if address_type == "street_only":
        # For street-only addresses, finding the street is the best possible result
        return "high"
    if fallback_used:
        return "low"
    return "medium"


def geocode_with_fallbacks(address_clean: str, address_type: str):
    """Try the address through a chain of transformations.
    
    Returns: (location, quality, method) or (None, 'failed', 'none')
    """
    # Attempt 1: original
    loc = geocode_in_malaga(address_clean)
    if loc:
        return loc, _classify_quality(loc, address_type, fallback_used=False), "original"
    
    # Attempt 2: typo fixes
    fixed = apply_typo_fixes(address_clean)
    if fixed != address_clean:
        loc = geocode_in_malaga(fixed)
        if loc:
            return loc, _classify_quality(loc, address_type, fallback_used=False), "typo_fix"
    
    # Attempt 3: expand name abbreviations
    expanded = expand_name_abbreviations(fixed)
    if expanded != fixed:
        loc = geocode_in_malaga(expanded)
        if loc:
            return loc, _classify_quality(loc, address_type, fallback_used=False), "expanded_names"
    
    # Attempt 4: strip district between number and postcode
    no_district = strip_district(expanded)
    if no_district != expanded:
        loc = geocode_in_malaga(no_district)
        if loc:
            return loc, _classify_quality(loc, address_type, fallback_used=False), "no_district"
    
    # Attempt 5: simplified (remove 'de la', 'del', ...)
    simplified = simplify_address(no_district)
    if simplified != address_clean:
        loc = geocode_in_malaga(simplified)
        if loc:
            return loc, _classify_quality(loc, address_type, fallback_used=False), "simplified"
    
    # Attempt 6: reduce house number range (13-11 → 13)
    reduced = reduce_house_range(address_clean)
    if reduced != address_clean:
        loc = geocode_in_malaga(reduced)
        if loc:
            return loc, _classify_quality(loc, address_type, fallback_used=False), "reduced_range"
    
    # Attempt 7: generic 'and'-split — works for any address with ' and '
    if " and " in address_clean.lower():
        street_a, street_b = split_intersection(address_clean)
        for variant in [street_a, simplify_address(street_a) if street_a else None,
                        street_b, simplify_address(street_b) if street_b else None]:
            if variant:
                loc = geocode_in_malaga(variant)
                if loc:
                    return loc, "low", "intersection_fallback"
    
    # Attempt 8: for house_number addresses, drop the number and try the street alone
    if address_type == "house_number":
        no_number = strip_house_number(address_clean)
        if no_number != address_clean:
            for variant in [no_number, simplify_address(no_number), expand_name_abbreviations(no_number)]:
                if variant:
                    loc = geocode_in_malaga(variant)
                    if loc:
                        return loc, "low", "street_only_fallback"
    
    return None, "failed", "none"

In [7]:
# Load existing cache (if any)
if CACHE_PATH.exists():
    with open(CACHE_PATH, "r", encoding="utf-8") as f:
        cache = json.load(f)
    print(f"Loaded {len(cache)} cached results")
else:
    cache = {}
    print("No cache found — starting fresh")


def save_cache():
    """Persist the cache to disk."""
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


# Drop cache entries that don't match current cleaned addresses
current_addresses = set(df["address_clean"])
stale = [k for k in cache if k not in current_addresses]
for k in stale:
    del cache[k]
if stale:
    print(f"Removed {len(stale)} stale cache entries")
    save_cache()

# Geocode addresses not yet in cache
to_process = [a for a in df["address_clean"] if a not in cache]
print(f"\nTotal addresses: {len(df)}")
print(f"In cache:         {len(df) - len(to_process)}")
print(f"To geocode now:   {len(to_process)}")

if to_process:
    print(f"Estimated time: ~{len(to_process) * 2.5 / 60:.1f} min")
    start = datetime.now()
    
    for i, addr in enumerate(to_process):
        addr_type = df.loc[df["address_clean"] == addr, "address_type"].iloc[0]
        loc, quality, method = geocode_with_fallbacks(addr, addr_type)
        
        if loc:
            cache[addr] = {
                "lat": loc.latitude,
                "lon": loc.longitude,
                "quality": quality,
                "method": method,
                "matched_address": loc.address,
                "match_class": loc.raw.get("class"),
                "match_type": loc.raw.get("type"),
            }
        else:
            cache[addr] = {
                "lat": None, "lon": None,
                "quality": "failed", "method": "none",
                "matched_address": None, "match_class": None, "match_type": None,
            }
        
        if (i + 1) % 25 == 0:
            elapsed = (datetime.now() - start).total_seconds()
            print(f"  [{i+1:3d}/{len(to_process)}]  elapsed {elapsed:.0f}s")
            save_cache()
    
    save_cache()
    print(f"\n✓ Done in {(datetime.now() - start).total_seconds():.0f}s")
else:
    print("\n✓ Nothing to do — all addresses already cached")

Loaded 478 cached results

Total addresses: 478
In cache:         478
To geocode now:   0

✓ Nothing to do — all addresses already cached


In [8]:
# Build a flat dataframe of results
df_geo = df.copy()
df_geo["lat"]               = [cache.get(a, {}).get("lat") for a in df_geo["address_clean"]]
df_geo["lon"]               = [cache.get(a, {}).get("lon") for a in df_geo["address_clean"]]
df_geo["geocoding_quality"] = [cache.get(a, {}).get("quality", "failed") for a in df_geo["address_clean"]]
df_geo["geocoding_method"]  = [cache.get(a, {}).get("method", "none") for a in df_geo["address_clean"]]
df_geo["matched_address"]   = [cache.get(a, {}).get("matched_address") for a in df_geo["address_clean"]]

# Overall quality distribution
print("=== Geocoding quality distribution ===")
print(df_geo["geocoding_quality"].value_counts())
print(f"\nSuccess rate: {(df_geo['geocoding_quality'] != 'failed').sum() / len(df_geo) * 100:.1f}%")

# Quality by address type
print("\n=== Quality by original address type ===")
print(pd.crosstab(df_geo["address_type"], df_geo["geocoding_quality"], margins=True))

# Methods used
print("\n=== Methods used (successful geocoding) ===")
print(df_geo.loc[df_geo["geocoding_quality"] != "failed", "geocoding_method"].value_counts())

=== Geocoding quality distribution ===
geocoding_quality
high      196
medium    132
low       102
failed     48
Name: count, dtype: int64

Success rate: 90.0%

=== Quality by original address type ===
geocoding_quality  failed  high  low  medium  All
address_type                                     
house_number           39   170   40     132  381
intersection            3     0   61       0   64
street_only             6    26    1       0   33
All                    48   196  102     132  478

=== Methods used (successful geocoding) ===
geocoding_method
original                         299
first_street_of_intersection      56
generic_and_fallback              33
simplified                        24
street_only_fallback               8
second_street_of_intersection      4
first_street_with_number           2
no_district                        1
expanded_names                     1
typo_fix                           1
first_street_simplified            1
Name: count, dtype: int64


## Failed addresses

Despite the fallback chain, ~9% of addresses could not be geocoded. The patterns below illustrate the remaining problem categories: typos absent from `MANUAL_TYPO_FIXES`, streets missing from OpenStreetMap, and ambiguous mixed formats. These are documented as limitations rather than fixed individually.

In [11]:
failed = df_geo[df_geo["geocoding_quality"] == "failed"]
print(f"Total failed addresses: {len(failed)}\n")

print("Sample (first 15):")
for _, row in failed.head(15).iterrows():
    print(f"  [{row['address_type']:14s}]  {row['address_clean']}")

print("\nFailed addresses by type:")
print(failed["address_type"].value_counts())

Total failed addresses: 48

Sample (first 15):
  [house_number  ]  Calle Lebrija, 6-8, Málaga, Spain
  [street_only   ]  Plaza Babel, Málaga, Spain
  [house_number  ]  Plaza Jose Begamin, 12, Málaga, Spain
  [street_only   ]  Avenida López de Vega, Málaga, Spain
  [house_number  ]  Avenida del Atabal, 15 Calle Juan de Ortega, Málaga, Spain
  [house_number  ]  Avenida Navarro Ledesma, 235, Málaga, Spain
  [house_number  ]  Calle Pindaro, 9 Entrada Parking Mercadona, Málaga, Spain
  [house_number  ]  Avenida Plutarco, 61 Antes de Avenidapermenides, Málaga, Spain
  [house_number  ]  Calle del Lingüista Lázaro Carreter, 2 and Calle Prof. Alfonso Pagonoski, Málaga, Spain
  [house_number  ]  Calle Plaza de la Inmaculada, 14, Málaga, Spain
  [house_number  ]  Calle Salvador Allende, 175, Málaga, Spain
  [house_number  ]  Rmoles, 14, Málaga, Spain
  [house_number  ]  Plaza Francisco López López, 5 En Rotonda, Málaga, Spain
  [house_number  ]  Ntiago Ramón y Cajal, 67, Málaga, Spain
  [intersec

## Duplicate coordinates

Different cleaned addresses sometimes return the same `(lat, lon)` from Nominatim. This can happen for two reasons:

- **Adjacent or nearby containers** that Nominatim could not resolve separately (e.g. two intersections sharing one street, or two house numbers on a short street that OSM only maps once)
- **Fallback-induced collisions** — when fallbacks strip the house number, two different `house_number` addresses on the same street collapse to the same street-centroid point

For the access analysis, points that fall in the same exact location effectively represent a single accessible spot. The list below flags these cases so they can be reviewed.

In [12]:
# Find addresses that share exact (lat, lon) with another address
df_ok = df_geo[df_geo["geocoding_quality"] != "failed"].copy()

# Round coords slightly to catch near-identical floats from different geocoder calls
df_ok["latlon_key"] = df_ok["lat"].round(5).astype(str) + "," + df_ok["lon"].round(5).astype(str)

dup_keys = df_ok["latlon_key"].value_counts()
dup_keys = dup_keys[dup_keys > 1]

print(f"Unique geocoded points: {df_ok['latlon_key'].nunique()}")
print(f"Coordinates shared by 2+ addresses: {len(dup_keys)}")
print(f"Total addresses affected: {dup_keys.sum()}\n")

# Show first 5 clusters
if len(dup_keys) > 0:
    print("=== First 5 duplicate-coordinate clusters ===\n")
    for key in dup_keys.head(5).index:
        cluster = df_ok[df_ok["latlon_key"] == key]
        lat, lon = cluster.iloc[0]["lat"], cluster.iloc[0]["lon"]
        print(f"({lat:.5f}, {lon:.5f}) — {len(cluster)} addresses:")
        for _, row in cluster.iterrows():
            print(f"   [{row['geocoding_quality']:6s} / {row['geocoding_method']:30s}]  {row['address_clean']}")
        print()

Unique geocoded points: 414
Coordinates shared by 2+ addresses: 11
Total addresses affected: 27

=== First 5 duplicate-coordinate clusters ===

(36.74576, -4.49173) — 5 addresses:
   [low    / first_street_of_intersection  ]  Calle Lope de Rueda and Calle Escritora Ana Maria Matute, Málaga, Spain
   [low    / first_street_of_intersection  ]  Calle Lope de Rueda and Avenidaarquitecto Luis Bono, Málaga, Spain
   [low    / first_street_of_intersection  ]  Calle Lope de Rueda and Camino del Orozco, Málaga, Spain
   [low    / first_street_of_intersection  ]  Calle Lope de Rueda and Calle Uranio, Málaga, Spain
   [low    / first_street_of_intersection  ]  Calle Lope de Rueda and Calle Juan Perez de Viedma, Málaga, Spain

(36.73060, -4.43293) — 4 addresses:
   [medium / original                      ]  Arroyo de los Angeles, 30, Málaga, Spain
   [low    / first_street_of_intersection  ]  Arroyo de los Angeles and Crisitino Martos, Málaga, Spain
   [low    / first_street_of_intersection  ]  Ar

In [9]:
# Check that all geocoded points are inside Málaga's bounding box
df_geo_ok = df_geo[df_geo["geocoding_quality"] != "failed"]

print(f"Successfully geocoded: {len(df_geo_ok)}")
print(f"  Latitude range:  {df_geo_ok['lat'].min():.5f} – {df_geo_ok['lat'].max():.5f}")
print(f"  Longitude range: {df_geo_ok['lon'].min():.5f} – {df_geo_ok['lon'].max():.5f}")

# Flag any outliers
outliers = df_geo_ok[
    (df_geo_ok["lat"] < MALAGA_BBOX[0][0]) | (df_geo_ok["lat"] > MALAGA_BBOX[1][0]) |
    (df_geo_ok["lon"] < MALAGA_BBOX[0][1]) | (df_geo_ok["lon"] > MALAGA_BBOX[1][1])
]
print(f"\nPoints outside Málaga bbox: {len(outliers)}")
if len(outliers) == 0:
    print("✓ All geocoded points are within the expected area.")

Successfully geocoded: 430
  Latitude range:  36.65329 – 36.75546
  Longitude range: -4.54948 – -4.34496

Points outside Málaga bbox: 0
✓ All geocoded points are within the expected area.


In [10]:
# Save the final geocoded dataset
df_geo.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Saved {len(df_geo)} rows to:")
print(f"  {OUTPUT_PATH}")
print(f"\nSummary:")
print(f"  Total addresses:        {len(df_geo)}")
print(f"  Successfully geocoded:  {(df_geo['geocoding_quality'] != 'failed').sum()}")
print(f"  Failed:                 {(df_geo['geocoding_quality'] == 'failed').sum()}")

df_geo.head(3)

Saved 478 rows to:
  c:\Users\user\Documents\Projects\malaga-textile-access\data\processed\containers_geocoded.csv

Summary:
  Total addresses:        478
  Successfully geocoded:  430
  Failed:                 48


,address_raw,address_type,address_clean,lat,lon,geocoding_quality,geocoding_method,matched_address
0,"Av. de la Paloma, 36-38, Carretera de Cádiz, 2...",house_number,"Avenida de la Paloma, 36-38, Carretera de Cádi...",36.700663,-4.441837,medium,simplified,"Avenida Paloma, Girón, Carretera de Cádiz, Mál..."
1,"Av. de la Paloma, 43, Carretera de Cádiz, 2900...",house_number,"Avenida de la Paloma, 43, Carretera de Cádiz, ...",36.702144,-4.445557,high,simplified,"43, Avenida Paloma, San Carlos Condote, Carret..."
2,"C. Conde de Guadalhorce, 6-8, Cruz de Humillad...",house_number,"Calle Conde de Guadalhorce, 6-8, Cruz de Humil...",36.714611,-4.442747,medium,original,"Calle Conde de Guadalhorce, Cruz del Humillade..."
